# Семинар 4. Сегментация и детекция

Сегодня разбираем на практике две задачи, где ответ сложнее одной метки на картинку: сегментацию (метка на каждый пиксель) и детекцию (набор прямоугольников с классами). Пройдём полный цикл на датасете Oxford-IIIT Pet: метрики, данные с масками, сеть кодировщик-декодировщик, обучение с логированием. Потом посмотрим, как пользоваться готовыми моделями из `torchvision`, и в конце на примере регуляризации свёрточных ядер обсудим, как ставить эксперимент, из которого можно сделать честный вывод.

В заданиях нужно дописывать код в местах с пропусками. Обучающие ячейки имеет смысл запускать на GPU; на CPU уменьшайте число эпох.


## Обозначения

- изображение: $\mathbf{x} \in \mathbb{R}^{3 \times H \times W}$;
- **сегментация**: ответ — маска $\mathbf{y} \in \{0,\dots,K-1\}^{H \times W}$, метка на каждый пиксель; сеть возвращает логиты $\mathbf{z} \in \mathbb{R}^{K \times H \times W}$;
- у нас будет **бинарная** сегментация ($K=2$: «питомец / фон»), поэтому логиты — один канал, вероятности даёт сигмоида $\sigma(z)$;
- **детекция**: ответ — набор прямоугольников $(x_1, y_1, x_2, y_2)$ с меткой класса и уверенностью $s \in [0,1]$;
- предсказанная маска при пороге $t$: $\hat{y}_{ij} = [\sigma(z_{ij}) > t]$, по умолчанию $t = 0.5$.

В коде: `logits` — выход сети до сигмоиды, `mask` — бинарная маска-ответ, `score` — уверенность детектора.


In [ ]:
# В Colab большинство пакетов уже стоят; локально раскомментируйте:
# !pip install torch torchvision albumentations torchmetrics scikit-image wandb comet_ml -q


In [ ]:
import os
import random
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

def set_global_seed(seed: int) -> None:
    """Фиксируем все источники случайности для воспроизводимости."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_global_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


## Часть 1: Метрики — почему не accuracy

В сегментации классы почти всегда несбалансированы: фон занимает бо́льшую часть кадра. Модель, которая отвечает «всюду фон», получает высокую попиксельную accuracy — и нулевую пользу. Поэтому основные метрики другие.

**IoU (Intersection over Union, мера Жаккарда)** — доля пересечения в объединении:

$$\mathrm{IoU}(\hat{Y}, Y) = \frac{|\hat{Y} \cap Y|}{|\hat{Y} \cup Y|} = \frac{TP}{TP + FP + FN}.$$

**Dice** — та же идея, но пересечение считается дважды:

$$\mathrm{Dice}(\hat{Y}, Y) = \frac{2\,|\hat{Y} \cap Y|}{|\hat{Y}| + |Y|} = \frac{2\,TP}{2\,TP + FP + FN}.$$

Обе метрики живут в $[0, 1]$, игнорируют true negatives (фон, угаданный как фон, не приносит очков) и связаны монотонной формулой $\mathrm{Dice} = \frac{2\,\mathrm{IoU}}{1 + \mathrm{IoU}}$. Dice — это в точности F1-мера, применённая к пикселям.

<div style="padding:12px; border-left: 4px solid #2E7D32; background: #E8F5E9;">
<b>Зачем нужен «мягкий» Dice</b><br/>
Метрика считается по бинарной маске, а обучать нужно по градиенту. Если вместо жёсткой маски подставить вероятности $p_{ij} = \sigma(z_{ij})$, получится дифференцируемый <b>soft-Dice</b>:
$$\mathrm{softDice} = \frac{2\sum_{ij} p_{ij}\, y_{ij} + \varepsilon}{\sum_{ij} p_{ij} + \sum_{ij} y_{ij} + \varepsilon},$$
и $1 - \mathrm{softDice}$ можно использовать как функцию потерь — в дополнение к кросс-энтропии.
</div>


### Задание 1: IoU и soft-Dice руками

Реализуйте обе функции и проверьте их на игрушечных масках. Подумайте перед запуском: чему равен IoU, если предсказание — сдвинутый на половину квадрат?


In [ ]:
def iou_binary(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """IoU для бинарных масок одного размера (значения 0/1)."""
    # ваш код: пересечение и объединение через операции над тензорами
    intersection = ...
    union = ...
    return (intersection + eps) / (union + eps)


def soft_dice(probs: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    """Дифференцируемый Dice: probs — вероятности (после сигмоиды), true_mask — 0/1."""
    # ваш код: формула softDice из ячейки выше
    num = ...
    den = ...
    return num / den


# --- проверки на игрушечных масках 4x4 ---
full = torch.ones(4, 4)
empty = torch.zeros(4, 4)
left = torch.zeros(4, 4); left[:, :2] = 1     # левая половина
right = torch.zeros(4, 4); right[:, 2:] = 1   # правая половина
mid = torch.zeros(4, 4); mid[:, 1:3] = 1      # средние два столбца

assert torch.isclose(iou_binary(full, full), torch.tensor(1.0), atol=1e-5)
assert iou_binary(left, right) < 1e-5                      # не пересекаются
assert torch.isclose(iou_binary(left, mid), torch.tensor(1/3), atol=1e-3)  # 4 клетки из 12
assert torch.isclose(soft_dice(left, mid), torch.tensor(0.5), atol=1e-3)   # Dice = 2*IoU/(1+IoU)

print('IoU(left, mid)  =', float(iou_binary(left, mid)))
print('Dice(left, mid) =', float(soft_dice(left, mid)))
print('Все проверки пройдены.')


<div style="padding:12px; border-left: 4px solid #455A64; background: #ECEFF1;">
<b>Итог после задания</b><br/>
IoU строже Dice (при том же качестве число меньше), но обе метрики упорядочивают модели одинаково. В статьях по сегментации чаще отчитывают mIoU — средний IoU по классам; в медицинских задачах — Dice.
</div>



## Часть 2: Данные — Oxford-IIIT Pet

Берём классический датасет: ~7400 фотографий кошек и собак, к каждой — trimap-маска (питомец / фон / граница). Мы сведём её к бинарной: питомец против всего остального.

Ключевое отличие от классификации: **аугментации применяются к картинке и маске синхронно**. Повернули картинку — обязаны так же повернуть маску, иначе разметка «отклеится». Для этого используем `albumentations`: она принимает пару (image, mask) и преобразует их одной случайной трансформацией.


In [ ]:
from torchvision.datasets.utils import download_and_extract_archive

DATA_ROOT = './oxford-iiit-pet'

def download_pets() -> None:
    resources = (
        ("https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz", "5c4f3ee8e5d25df40f4fd59a7f44e54c"),
        ("https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz", "95a8c909bbe2e81eed6a22bccdf3f68f"),
    )
    for url, md5 in resources:
        download_and_extract_archive(url, download_root=DATA_ROOT, md5=md5)

if not os.path.exists(DATA_ROOT):
    download_pets()


In [ ]:
class PetsSegmentation(torch.utils.data.Dataset):
    """Oxford-IIIT Pet как задача бинарной сегментации.

    В trimap: 1 — питомец, 2 — фон, 3 — граница.
    Мы объявляем питомцем классы {1, 3}, фон — класс 2.
    """

    def __init__(self, root: str, split: str = 'trainval', transforms=None):
        self._images_folder = os.path.join(root, 'images')
        self._segs_folder = os.path.join(root, 'annotations', 'trimaps')
        split_file = os.path.join(root, 'annotations', f'{split}.txt')
        with open(split_file) as f:
            self._ids = [line.split()[0] for line in f]
        self.transforms = transforms

    def __len__(self):
        return len(self._ids)

    def __getitem__(self, idx):
        name = self._ids[idx]
        image = np.array(Image.open(os.path.join(self._images_folder, name + '.jpg')).convert('RGB'))
        trimap = np.array(Image.open(os.path.join(self._segs_folder, name + '.png')))
        mask = (trimap != 2).astype(np.float32)  # питомец + граница = 1, фон = 0

        if self.transforms is not None:
            out = self.transforms(image=image, mask=mask)
            image, mask = out['image'], out['mask']
        return image, mask


In [ ]:
# Нормировка ImageNet — та же, на которой обучался будущий энкодер (см. лекцию 3 про нормировку входа)
CHANNEL_MEAN = (0.485, 0.456, 0.406)
CHANNEL_STD = (0.229, 0.224, 0.225)

train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1), rotate=(-10, 10), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.4),
    A.Normalize(mean=CHANNEL_MEAN, std=CHANNEL_STD),
    ToTensorV2(),
])

test_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=CHANNEL_MEAN, std=CHANNEL_STD),
    ToTensorV2(),
])

train_dataset = PetsSegmentation(DATA_ROOT, split='trainval', transforms=train_transforms)
test_dataset = PetsSegmentation(DATA_ROOT, split='test', transforms=test_transforms)
len(train_dataset), len(test_dataset)


In [ ]:
BATCH_SIZE = 32

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=(device == 'cuda'), drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=2 * BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=(device == 'cuda'))


In [ ]:
def denormalize(image: torch.Tensor) -> np.ndarray:
    """Обратная нормировка для отрисовки."""
    image = image.permute(1, 2, 0).cpu().numpy()
    image = image * np.array(CHANNEL_STD) + np.array(CHANNEL_MEAN)
    return np.clip(image, 0, 1)

def show_batch(images, masks, preds=None, n=4, title=''):
    cols = 2 if preds is None else 3
    fig, axs = plt.subplots(n, cols, figsize=(3 * cols, 3 * n))
    fig.suptitle(title)
    for i in range(n):
        axs[i, 0].imshow(denormalize(images[i])); axs[i, 0].set_title('изображение')
        axs[i, 1].imshow(masks[i].cpu(), cmap='gray'); axs[i, 1].set_title('маска')
        if preds is not None:
            axs[i, 2].imshow(preds[i], cmap='gray'); axs[i, 2].set_title('предсказание')
        for ax in axs[i]:
            ax.axis('off')
    plt.tight_layout(); plt.show()

images, masks = next(iter(train_dataloader))
show_batch(images, masks, title='Аугментированные примеры из train')


<div style="padding:12px; border-left: 4px solid #FFA000; background: #FFF8E1;">
<b>Проверьте себя</b><br/>
Запустите предыдущую ячейку несколько раз: маска обязана «ездить» вместе с картинкой. Если бы мы применяли <code>torchvision.transforms</code> к картинке и маске независимо, каждая получила бы <i>свою</i> случайную трансформацию — и разметка бы разъехалась.
</div>



## Часть 3: Модель — кодировщик-декодировщик

Классическая схема из лекции 7: **кодировщик** сжимает изображение в признаки (это обычная классификационная CNN без головы), **декодировщик** разворачивает признаки обратно до размера картинки.

Кодировщик не будем обучать с нуля — возьмём VGG11 с BatchNorm, предобученную на ImageNet, и **заморозим** её веса (transfer learning, как в лекции 6). Обучаться будет только декодировщик.

Для увеличения разрешения в декодировщике есть два основных инструмента — интерполяция (`Upsample` + обычная свёртка) и транспонированная свёртка (`ConvTranspose2d`, лекция 7). Посмотрим на интерполяцию на игрушечном тензоре.


In [ ]:
def show_tensor(tensor, text, axis=None):
    if axis is None:
        axis = plt.gca()
    t = tensor.squeeze().detach().cpu().numpy()
    im = axis.imshow(t, cmap='viridis')
    for (j, i), v in np.ndenumerate(t):
        axis.text(i, j, f'{v:.0f}', ha='center', va='center', color='white', fontsize=8)
    axis.set_xticks([]); axis.set_yticks([])
    axis.set_title(text)

ex = torch.randint(0, 100, (2, 2)).view(1, 1, 2, 2).float()

fig, axs = plt.subplots(1, 4, figsize=(14, 3.5))
show_tensor(ex, 'вход 2×2', axs[0])
show_tensor(nn.Upsample(scale_factor=2, mode='nearest')(ex), 'nearest', axs[1])
show_tensor(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)(ex), 'bilinear', axs[2])
show_tensor(nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2)(ex), 'ConvTranspose2d (случайные веса)', axs[3])
plt.tight_layout(); plt.show()


In [ ]:
from torchvision.models import vgg11_bn, VGG11_BN_Weights

encoder = vgg11_bn(weights=VGG11_BN_Weights.IMAGENET1K_V1).features
for p in encoder.parameters():
    p.requires_grad = False

# 224x224 -> после 5 пулингов 7x7, 512 каналов
with torch.no_grad():
    print('выход энкодера:', encoder(torch.zeros(1, 3, 224, 224)).shape)


In [ ]:
class UpsamplingBlock(nn.Module):
    """Один шаг декодировщика: x2 по разрешению + свёртка + Dropout + ReLU + BatchNorm."""

    def __init__(self, channel_in, channel_out, dropout_rate=0.3):
        super().__init__()
        self.conv = nn.Conv2d(channel_in, channel_out, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(dropout_rate)
        self.activation = nn.ReLU()
        self.batch_norm = nn.BatchNorm2d(channel_out)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2)
        x = self.conv(x)
        x = self.dropout(x)
        x = self.activation(x)
        return self.batch_norm(x)


class FCNDecoder(nn.Module):
    """Каскад из пяти UpsamplingBlock: 7x7x512 -> 224x224x32."""

    def __init__(self):
        super().__init__()
        self._channel_map = [512, 512, 256, 128, 64, 32]
        self.blocks = nn.ModuleList([
            UpsamplingBlock(c_in, c_out)
            for c_in, c_out in zip(self._channel_map[:-1], self._channel_map[1:])
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x


class FCN32(nn.Module):
    """Кодировщик-декодировщик для бинарной сегментации (без skip-связей).

    classifier — свёртка 1x1: бинарная логистическая регрессия на каждый пиксель.
    """

    def __init__(self, encoder, decoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.classifier = classifier

    def forward(self, x):
        x = self.encoder(x)     # [B, 3, 224, 224] -> [B, 512, 7, 7]
        x = self.decoder(x)     # -> [B, 32, 224, 224]
        return self.classifier(x).squeeze(1)  # -> [B, 224, 224], логиты


def make_model():
    """Свежая модель с замороженным энкодером (для честных сравнений)."""
    enc = vgg11_bn(weights=VGG11_BN_Weights.IMAGENET1K_V1).features
    for p in enc.parameters():
        p.requires_grad = False
    return FCN32(enc, FCNDecoder(), nn.Conv2d(32, 1, kernel_size=1)).to(device)


set_global_seed(42)
model = make_model()

with torch.no_grad():
    print('выход модели:', model(torch.zeros(2, 3, 224, 224).to(device)).shape)


In [ ]:
def print_params_count(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Всего параметров:            {total:,}')
    print(f'Обучаемых параметров:        {trainable:,}')
    print(f'Доля обучаемых:              {trainable / total:.3f}')

print_params_count(model)


### Задание 2: где skip?

У нашей `FCN32` есть архитектурная проблема, о которой прямо говорила лекция 7: вся пространственная информация проходит через узкое место $7 \times 7$, и мелкие детали (уши, лапы, граница шерсти) декодировщику приходится «выдумывать».

**Вопрос.** Как U-Net решает эту проблему? Что именно нужно прокинуть из энкодера в декодировщик и как объединить с текущим тензором?



## Часть 4: Обучение и логирование

Функция потерь — `BCEWithLogitsLoss` (бинарная кросс-энтропия по логитам; сигмоида спрятана внутрь ради численной устойчивости — тот же приём, что с softmax и кросс-энтропией в классификации). Метрика — IoU из `torchmetrics`.

Прежде чем писать цикл обучения, решим, **что мы хотим видеть во время обучения**:

1. Метрики на обучении и на тесте — чтобы вовремя заметить переобучение (test растёт, train падает) или выход на плато.
2. Состояние оптимизации: текущий learning rate, если работает scheduler.
3. Распределения весов по слоям — чтобы ловить неустойчивость обучения.
4. Промежуточные результаты модели: в нашей задаче это предсказанные маски поверх картинок. Числа не расскажут, *что именно* сеть ошибается на ушах и хвостах, — картинки расскажут.

Пункты 1–2 можно собрать и руками в словарь `history` (мы так и сделаем — для графиков в ноутбуке), но пункты 3–4 удобны только в специальном инструменте — W&B или comet. Ниже — обёртка, работающая с обоими; выберите логгер флагом `USE_COMET` и включите `USE_LOGGING`, когда у вас есть аккаунт (в Colab ключ удобно хранить в secrets).


In [ ]:
# Логирование в W&B / comet. По умолчанию выключено, чтобы ноутбук
# запускался без аккаунта; на реальном проекте держите включённым.
USE_LOGGING = False
USE_COMET = False   # False -> wandb, True -> comet_ml

if USE_LOGGING:
    import wandb
    import comet_ml
    # from google.colab import userdata   # ключи в Colab: Secrets слева


def logger_start(project_name, run_name):
    if not USE_LOGGING:
        return
    if USE_COMET:
        experiment = comet_ml.start(
            api_key=userdata.get('COMET_API_KEY'),
            project_name=project_name, online=True,
        )
        experiment.set_name(run_name)
    else:
        wandb.init(project=project_name, name=run_name, mode='online')


def log_metrics(d: dict, step: int):
    if not USE_LOGGING:
        return
    if USE_COMET:
        comet_ml.get_running_experiment().log_metrics(d, step=step)
    else:
        wandb.log(d, step=step)


def log_hist(arr, name: str, step: int):
    if not USE_LOGGING:
        return
    if USE_COMET:
        comet_ml.get_running_experiment().log_histogram_3d(arr, name=name, step=step)
    else:
        wandb.log({name: wandb.Histogram(arr)}, step=step)


def logger_end():
    if not USE_LOGGING:
        return
    if USE_COMET:
        comet_ml.get_running_experiment().end()
    else:
        wandb.finish()


Логирование **картинок с масками**: и W&B, и comet умеют накладывать маски на изображение прямо в интерфейсе, с переключением слоёв «предсказание / разметка». Выводом в консоль или самодельным csv такого не получить.

Документация: [wandb image overlays](https://docs.wandb.ai/models/track/log/media#image-overlays), [comet log image with regions](https://www.comet.com/docs/v2/guides/experiment-management/log-data/images/#log-image-with-regions).


In [ ]:
from skimage import measure


def _make_mask_points(binary_mask):
    """Контуры бинарной маски как список полигонов (нужно для comet)."""
    contours = measure.find_contours(binary_mask, 0.5)
    points = []
    for contour in contours:
        contour = np.flip(contour, axis=1)
        points.append(contour.ravel().tolist())
    return points


def log_images(img, ground_truth, pred_mask, img_name, step, threshold=0.0):
    """Логирует картинку с двумя масками-слоями: предсказание и разметка."""
    if not USE_LOGGING:
        return
    img = denormalize(img.detach().cpu())
    ground_truth = ground_truth.detach().cpu().numpy()
    pred_mask = pred_mask.detach().cpu().numpy()

    class_labels = {-1: 'background', 1: 'pet'}
    pred_mask = np.where(pred_mask <= threshold, -1, 1)
    ground_truth = np.where(ground_truth <= threshold, -1, 1)

    if USE_COMET:
        annotations = []
        for name, mask in [('prediction', pred_mask), ('ground_truth', ground_truth)]:
            pts = _make_mask_points(mask == 1) or [[0, 0, 0, 0, 0, 0]]
            annotations.append({'name': name, 'data': [{'label': 'pet', 'points': pts}]})
        comet_ml.get_running_experiment().log_image(
            image_data=img, name=img_name, annotations=annotations, step=step)
    else:
        image = wandb.Image(img, masks={
            'predictions': {'mask_data': pred_mask, 'class_labels': class_labels},
            'ground_truth': {'mask_data': ground_truth, 'class_labels': class_labels},
        })
        wandb.log({img_name: image}, step=step)


In [ ]:
from torchmetrics import JaccardIndex

criterion = nn.BCEWithLogitsLoss()
metric = JaccardIndex(task='binary').to(device)

lr = 1e-3
weight_decay = 1e-4
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)


In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_iou, n_batches = 0.0, 0.0, 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        total_loss += loss_fn(logits, masks).item()
        total_iou += metric(logits.sigmoid() > 0.5, masks.bool()).item()
        n_batches += 1
    return total_loss / n_batches, total_iou / n_batches


def train_model(model, loss_fn, optimizer, scheduler, num_epochs=3, extra_loss=None,
                run_name=None, n_images_to_log=8):
    """extra_loss(logits, masks) — необязательное слагаемое (например, 1 - softDice)."""
    if run_name is not None:
        logger_start(project_name='cu-dl-week4', run_name=run_name)

    # фиксированные картинки из теста — чтобы следить за одними и теми же примерами
    log_ids = np.random.RandomState(0).choice(len(test_dataset), n_images_to_log)
    images_to_log = torch.stack([test_dataset[i][0] for i in log_ids])
    masks_to_log = torch.stack([test_dataset[i][1] for i in log_ids])

    history = {'train_loss': [], 'test_loss': [], 'test_iou': []}
    for epoch in range(num_epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0
        for images, masks in train_dataloader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, masks)
            if extra_loss is not None:
                loss = loss + extra_loss(logits, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
        scheduler.step()

        test_loss, test_iou = evaluate(model, test_dataloader, loss_fn)
        history['train_loss'].append(epoch_loss / n_batches)
        history['test_loss'].append(test_loss)
        history['test_iou'].append(test_iou)

        # --- логирование: метрики, lr, гистограммы весов, картинки с масками ---
        log_metrics({'train/loss': history['train_loss'][-1],
                     'test/loss': test_loss, 'test/IoU': test_iou,
                     'lr': scheduler.get_last_lr()[0]}, step=epoch)
        for block_num, block in enumerate(model.decoder.blocks):
            log_hist(block.conv.weight.data.cpu().numpy(),
                     f'weights/upsampling_block_{block_num}', step=epoch)
        if USE_LOGGING:
            model.eval()
            with torch.no_grad():
                preds = model(images_to_log.to(device))
            for i in range(n_images_to_log):
                log_images(images_to_log[i], masks_to_log[i], preds[i], f'image_{i}', step=epoch)

        print(f'эпоха {epoch + 1}: train loss {history["train_loss"][-1]:.4f} | '
              f'test loss {test_loss:.4f} | test IoU {test_iou:.4f}')

    if run_name is not None:
        logger_end()
    return history


In [ ]:
set_global_seed(42)
history = train_model(model, criterion, optimizer, scheduler, num_epochs=3,
                      run_name='BCE-baseline')


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(history['train_loss'], label='train'); axs[0].plot(history['test_loss'], label='test')
axs[0].set_title('loss'); axs[0].set_xlabel('эпоха'); axs[0].legend()
axs[1].plot(history['test_iou']); axs[1].set_title('test IoU'); axs[1].set_xlabel('эпоха')
plt.show()


In [ ]:
model.eval()
images, masks = next(iter(test_dataloader))
with torch.no_grad():
    preds = (model(images.to(device)).sigmoid() > 0.5).cpu()
show_batch(images, masks, preds, title='Предсказания после обучения')


За 3 эпохи с замороженным энкодером маски получаются грубыми — это нормально. Чтобы увидеть, на что способна **эта же архитектура** при полноценном обучении, загрузим готовый чекпоинт (обучен на этой конфигурации в семинаре, лежит на HuggingFace).


In [ ]:
USE_PRETRAINED = True

if USE_PRETRAINED:
    if not os.path.exists('./segmentation'):
        !git clone https://huggingface.co/3145tttt/segmentation
    model_full = make_model()
    model_full.load_state_dict(torch.load('segmentation/model.pth', map_location=device)['model'])
    model_full.eval()

    with torch.no_grad():
        preds_full = (model_full(images.to(device)).sigmoid() > 0.5).cpu()
    show_batch(images, masks, preds_full, title='Предсказания дообученного чекпоинта')

    test_loss_full, test_iou_full = evaluate(model_full, test_dataloader, criterion)
    print(f'чекпоинт: test loss {test_loss_full:.4f} | test IoU {test_iou_full:.4f}')


### Задание 3: добавляем Dice к лоссу

Кросс-энтропия оптимизирует каждый пиксель независимо и «не знает» про несбалансированность классов; soft-Dice смотрит на маску целиком. На практике их часто складывают.

1. Соберите слагаемое `dice_loss(logits, masks) = 1 - soft_dice(logits.sigmoid(), masks)` из вашей функции из Задания 1.
2. Создайте свежую модель (тот же `set_global_seed(42)`!) и обучите её с `extra_loss=dice_loss` те же 3 эпохи.
3. Сравните `test_iou` двух запусков. Что изменилось и почему сравнение честное?


In [ ]:
def dice_loss(logits, masks):
    # ваш код: слагаемое 1 - softDice по вероятностям сигмоиды
    return ...

set_global_seed(42)
model_dice = make_model()

optimizer_dice = torch.optim.AdamW([p for p in model_dice.parameters() if p.requires_grad],
                                   lr=lr, weight_decay=weight_decay)
scheduler_dice = torch.optim.lr_scheduler.ExponentialLR(optimizer_dice, gamma=0.9)

history_dice = train_model(model_dice, criterion, optimizer_dice, scheduler_dice,
                           num_epochs=3, extra_loss=dice_loss, run_name='BCE-plus-Dice')

plt.plot(history['test_iou'], label='BCE')
plt.plot(history_dice['test_iou'], label='BCE + Dice')
plt.xlabel('эпоха'); plt.ylabel('test IoU'); plt.legend(); plt.show()


<div style="padding:12px; border-left: 4px solid #455A64; background: #ECEFF1;">
<b>Итог после задания</b><br/>
Сравнение честное, потому что зафиксировано всё, кроме одного фактора: тот же сид, та же архитектура, тот же оптимизатор и число эпох — отличается только функция потерь. Запомните этот принцип: он главный герой финальной части семинара.
</div>



## Часть 5: Готовые модели — сегментация и детекция за пять строк

Обучать с нуля нужно далеко не всегда: в `torchvision` лежат модели, обученные на COCO. Сначала — семантическая сегментация (FCN на ResNet-50, 21 класс PASCAL VOC).


In [ ]:
from torchvision.models.segmentation import fcn_resnet50, FCN_ResNet50_Weights

seg_model = fcn_resnet50(weights=FCN_ResNet50_Weights.DEFAULT).to(device).eval()

url = "https://github.com/pytorch/hub/raw/master/images/deeplab1.png"
urllib.request.urlretrieve(url, "example.png")
input_image = Image.open("example.png").convert("RGB")

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CHANNEL_MEAN, std=CHANNEL_STD),
])
batch = preprocess(input_image).unsqueeze(0).to(device)

with torch.no_grad():
    out = seg_model(batch)['out'][0]          # [21, H, W] — логиты классов на каждый пиксель
class_map = out.argmax(0).cpu()

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(input_image); axs[0].set_title('вход'); axs[0].axis('off')
axs[1].imshow(class_map, cmap='tab20'); axs[1].set_title('классы пикселей'); axs[1].axis('off')
plt.show()


Теперь детекция — Faster R-CNN из лекции 8. Обратите внимание на формат выхода: это уже не тензор фиксированной формы, а **список словарей** — по одному на картинку, с переменным числом боксов.


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.utils import draw_bounding_boxes

weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
det_model = fasterrcnn_resnet50_fpn_v2(weights=weights).to(device).eval()
COCO_CLASSES = weights.meta['categories']

# детектору нормировка не нужна — только тензор [0, 1]
det_batch = transforms.ToTensor()(input_image).unsqueeze(0).to(device)

with torch.no_grad():
    detections = det_model(det_batch)[0]

print('ключи выхода:', detections.keys())
print('всего боксов:', len(detections['boxes']))
print('первые уверенности:', detections['scores'][:5].tolist())


In [ ]:
def show_detections(image_tensor, det, score_threshold=0.0, title=''):
    keep = det['scores'] > score_threshold
    img_uint8 = (image_tensor * 255).byte().cpu()
    labels = [f"{COCO_CLASSES[l]} {s:.2f}"
              for l, s in zip(det['labels'][keep], det['scores'][keep])]
    boxed = draw_bounding_boxes(img_uint8, det['boxes'][keep].cpu(), labels=labels, width=4)
    plt.figure(figsize=(8, 6))
    plt.imshow(boxed.permute(1, 2, 0)); plt.axis('off'); plt.title(title); plt.show()

show_detections(det_batch[0], detections, score_threshold=0.0,
                title='Все кандидаты — без порога уверенности')


### Задание 4: порог уверенности и NMS

Картинка выше — типичный «сырой» выход детектора: десятки перекрывающихся боксов на один объект.

1. Подберите `score_threshold`, при котором остаются только настоящие объекты. Что произойдёт, если задрать порог слишком высоко?
2. Порог не решает всё: два уверенных бокса могут описывать один объект. Примените non-maximum suppression (лекция 8) через `torchvision.ops.nms` и сравните число оставшихся боксов при `iou_threshold` 0.9, 0.5 и 0.2.


In [ ]:
from torchvision.ops import nms

score_threshold = 0.5  # подберите: посмотрите также на 0.3 и 0.8
keep_score = detections['scores'] > score_threshold
boxes = detections['boxes'][keep_score]
scores = detections['scores'][keep_score]
labels = detections['labels'][keep_score]

for iou_thr in [0.9, 0.5, 0.2]:
    keep_nms = nms(boxes, scores, iou_threshold=iou_thr)
    print(f'iou_threshold={iou_thr}: осталось {len(keep_nms)} боксов из {len(boxes)}')

keep_nms = nms(boxes, scores, iou_threshold=0.5)
filtered = {'boxes': boxes[keep_nms], 'scores': scores[keep_nms], 'labels': labels[keep_nms]}
show_detections(det_batch[0], filtered, title='После порога и NMS')


<div style="padding:12px; border-left: 4px solid #455A64; background: #ECEFF1;">
<b>Итог после задания</b><br/>
Пара «порог уверенности + NMS» — обязательный постпроцессинг любого детектора. А метрика mAP из лекции 8 устроена так, что перебирает <i>все</i> пороги сразу: строит precision–recall кривую по уверенностям и берёт площадь под ней, усредняя по классам и порогам IoU.
</div>



## Часть 6 (финал): как ставить честный эксперимент

До сих пор мы применяли известные методы; теперь ситуация исследователя: **вот статья с правдоподобной идеей и скромным эффектом — как проверить, работает ли она?**

Идея (Wang et al., *Orthogonal Convolutional Neural Networks*, CVPR 2020): добавить штраф на свёрточные ядра

$$\mathcal{L}_{orth} = \big\| K * K - I\delta \big\|,$$

то есть потребовать, чтобы свёртка ядра с самим собой давала (почти) дельта-функцию — фильтры ортогональны. Мотивация: сохранение нормы сигнала, значит градиенты не затухают и не взрываются, а фильтры не дублируют друг друга. Это регуляризационный родственник ортогональной инициализации из лекции 3 — только действует всё обучение.


In [ ]:
def deconv_orth_dist(kernel: torch.Tensor, stride: int = 2, padding: int = 1) -> torch.Tensor:
    """Штраф ||K * K - I·delta|| из авторской имплементации Orthogonal CNN."""
    o_c, i_c, w, h = kernel.shape
    output = torch.conv2d(kernel, kernel, stride=stride, padding=padding)
    target = torch.zeros((o_c, o_c, output.shape[-2], output.shape[-1]), device=kernel.device)
    ct = int(np.floor(output.shape[-1] / 2))
    target[:, :, ct, ct] = torch.eye(o_c, device=kernel.device)
    return torch.norm(output - target)


Прежде чем писать цикл обучения — главный вопрос: **как должен быть устроен эксперимент, чтобы вывод был честным?** Три принципа:

1. **Эффект должен иметь шанс проявиться.** Регуляризатор проверяют на модели, склонной к переобучению: без предобучения, на небольшом датасете. Если сеть и так недообучена — регуляризация только помешает, и мы «опровергнем» метод, не проверив его.
2. **Эффект должен быть измерим.** Заранее решаем, на какую метрику смотрим (test accuracy) и какая разница нас убедит. Полпроцента на одном сиде не убеждает — нужны несколько сидов.
3. **Меняется только один фактор.** Все конфигурации получают одинаковую архитектуру, данные, оптимизатор, расписание и сиды; различается только наличие и вес $\mathcal{L}_{orth}$.

Конфигурация по мотивам семинара ММП: `mobilenet_v3_small` **без** предобучения на CIFAR-10 (половина train — чтобы переобучение наступало быстрее).


In [ ]:
from torchvision.models import mobilenet_v3_small
from torchvision.datasets import CIFAR10

set_global_seed(42)

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CHANNEL_MEAN, std=CHANNEL_STD),
])
cifar_full = CIFAR10('.', train=True, transform=cifar_transform, download=True)
cifar_train, cifar_test = torch.utils.data.random_split(cifar_full, (25000, 25000))

cifar_train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True, num_workers=2)
cifar_test_loader = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=2)


def make_mobilenet():
    m = mobilenet_v3_small(weights=None)
    m.classifier[3] = nn.Linear(1024, 10)
    return m.to(device)


def collect_conv_kernels(model):
    """Ядра всех 'квадратных' свёрток (kernel_size >= 2), к которым применим штраф."""
    return [m.weight for m in model.modules()
            if isinstance(m, nn.Conv2d) and m.weight.shape[-1] >= 2 and m.groups == 1]


In [ ]:
def train_cifar(model, num_epochs=5, orth_weight=0.0, weight_decay=1e-4, seed=42):
    """Один запуск: возвращает test accuracy по эпохам."""
    set_global_seed(seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.98)
    kernels = collect_conv_kernels(model)
    accs = []

    for epoch in range(num_epochs):
        model.train()
        for x, y in cifar_train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            if orth_weight > 0:
                loss = loss + orth_weight * sum(deconv_orth_dist(k) for k in kernels)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval(); correct = total = 0
        with torch.no_grad():
            for x, y in cifar_test_loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.numel()
        accs.append(correct / total)
        print(f'  эпоха {epoch + 1}: test acc {accs[-1]:.4f}')
    return accs


### Задание 5: спроектируйте и проведите эксперимент

Ниже — заготовка на три конфигурации. Прежде чем запускать, письменно (в ячейке ниже) ответьте:

1. Почему в конфигурациях зафиксирован сид и чем плох вывод по одному сиду? Сколько сидов взяли бы вы?
2. Зачем нужна конфигурация C (орто-штраф **вместо** weight decay, а не вместе)? На какой вопрос она отвечает, на который не отвечает пара A/B?
3. Какой результат вы **ожидаете**, учитывая, что в современном сетапе (нормальная инициализация + BatchNorm + weight decay) многие эффекты таких регуляризаторов уже «съедены»?

Затем запустите (на GPU это ~10–15 минут; на CPU — уменьшите `num_epochs` до 2 и число батчей).



In [ ]:
EPOCHS = 5  # на CPU поставьте 2

print('A. Бейзлайн: weight decay, без орто-штрафа')
acc_a = train_cifar(make_mobilenet(), num_epochs=EPOCHS, orth_weight=0.0, weight_decay=1e-4)

print('B. Weight decay + орто-штраф')
acc_b = train_cifar(make_mobilenet(), num_epochs=EPOCHS, orth_weight=1e-2, weight_decay=1e-4)

print('C. Только орто-штраф (без weight decay)')
acc_c = train_cifar(make_mobilenet(), num_epochs=EPOCHS, orth_weight=1e-2, weight_decay=0.0)

plt.plot(acc_a, label='A: WD'); plt.plot(acc_b, label='B: WD + orth'); plt.plot(acc_c, label='C: orth')
plt.xlabel('эпоха'); plt.ylabel('test accuracy'); plt.legend(); plt.show()



В статье заявлен выигрыш ~0.5–1 п.п. на классификации — в основном на недорегуляризованных сетях и малых данных. В хорошо настроенном современном сетапе эффект часто исчезает. Это <b>не</b> значит, что идея плохая: спектральные ограничения на веса всерьёз работают в GAN и в задачах робастности. Вывод, который стоит унести: «в статье работает» и «нужно в вашей задаче» — разные утверждения, и различить их может только честно поставленный эксперимент.
</div>


## Итоги

### Что мы сделали

- Разобрали метрики сегментации: IoU и Dice, их связь и «мягкую» дифференцируемую версию для лосса.
- Собрали датасет с масками и синхронными аугментациями.
- Построили кодировщик-декодировщик на замороженном VGG11 и обучили его; обсудили, зачем U-Net нужны skip-связи.
- Сравнили BCE и BCE+Dice в честном сравнении с фиксированным сидом.
- Применили готовые модели: семантическая сегментация и Faster R-CNN, порог уверенности и NMS.
- Спроектировали эксперимент по проверке ортогональной регуляризации — с контролем всех факторов, кроме одного.

### Чек-лист «сегментация не учится»

1. Маска и картинка аугментируются **одной** трансформацией? Нарисуйте батч и проверьте глазами.
2. Нормировка входа совпадает с той, на которой обучался энкодер?
3. Лосс считается по **логитам** (`BCEWithLogitsLoss`), а не по вероятностям?
4. Размер выхода сети совпадает с размером маски?
5. Метрика падает при `model.train()` на тесте? Проверьте `model.eval()` — BatchNorm и Dropout ведут себя по-разному (семинар 1).

### Мини-домашка

- Доделайте skip-connections из Задания 2 и сравните IoU с бейзлайном — в том же протоколе, что и Задание 3.
- Замените энкодер на ResNet-18: какие слои снимать и как изменится декодировщик?
- Для детектора: посчитайте precision и recall на выбранной картинке вручную при трёх порогах уверенности — и убедитесь, что двигая порог, вы едете по precision–recall кривой из лекции 8.
